In [ ]:
#imports
import numpy as np



# Unit 2, Notebook 2: The Symbolic Derivative

In the previous notebook, we built mathematical objects that know how to evaluate themselves. Now, we are going to add a new method to our base class: `.derive(variable)`.

Instead of giving us a number, `.derive()` will output a brand-new **Expression object** that represents the derivative.


In [ ]:
# We are updating our base classes from the previous notebook.
# In a real environment, we would import them, but we will redefine them here for clarity.

class Expression:
    def evaluate(self, context):
        pass
        
    def derive(self, var_name):
        raise NotImplementedError("Each subclass must implement derive()")
        
    def __add__(self, other):
        return Add(self, other)
        
    def __mul__(self, other):
        return Multiply(self, other)



```python
# We are updating our base classes from the previous notebook.
# In a real environment, we would import them, but we will redefine them here for clarity.

class Expression:
    def evaluate(self, context):
        pass
        
    def derive(self, var_name):
        raise NotImplementedError("Each subclass must implement derive()")
        
    def __add__(self, other):
        return Add(self, other)
        
    def __mul__(self, other):
        return Multiply(self, other)

```

### **[Markdown Cell]**

## 1. The Derivative of Constants and Variables

The rules for basic derivatives are simple:

* The derivative of a constant is always $0$.
* The derivative of a variable with respect to itself is $1$.
* The derivative of a variable with respect to a *different* variable is $0$.

Let's implement this logic into our classes.

### **[Code Cell]**

```python
class Constant(Expression):
    def __init__(self, value):
        self.value = value
        
    def __str__(self): return str(self.value)
    
    def derive(self, var_name):
        # The derivative of any constant is 0
        return Constant(0)

class Variable(Expression):
    def __init__(self, name):
        self.name = name
        
    def __str__(self): return self.name
    
    def derive(self, var_name):
        # If we derive 'x' with respect to 'x', the slope is 1.
        # If we derive 'y' with respect to 'x', the slope is 0.
        if self.name == var_name:
            return Constant(1)
        else:
            return Constant(0)

```

### **[Markdown Cell]**

## 2. The Sum Rule

In calculus, the derivative of $f(x) + g(x)$ is simply $f'(x) + g'(x)$.

In code, this means our `Add` class just asks its left child for its derivative, asks its right child for its derivative, and adds them together!

### **[Code Cell]**

```python
class Add(Expression):
    def __init__(self, left, right):
        self.left = left if isinstance(left, Expression) else Constant(left)
        self.right = right if isinstance(right, Expression) else Constant(right)
        
    def __str__(self): return f"({self.left} + {self.right})"
    
    def derive(self, var_name):
        # The Sum Rule: derive the left, derive the right, add them.
        return Add(self.left.derive(var_name), self.right.derive(var_name))

```

### **[Markdown Cell]**

## 3. The Product Rule

This is where the magic happens. The derivative of $f(x) \cdot g(x)$ is:


$$f'(x)g(x) + f(x)g'(x)$$

Let's translate that exact formula into our `Multiply` class.

### **[Code Cell]**

```python
class Multiply(Expression):
    def __init__(self, left, right):
        self.left = left if isinstance(left, Expression) else Constant(left)
        self.right = right if isinstance(right, Expression) else Constant(right)
        
    def __str__(self): return f"({self.left} * {self.right})"
    
    def derive(self, var_name):
        # The Product Rule: (left' * right) + (left * right')
        term1 = Multiply(self.left.derive(var_name), self.right)
        term2 = Multiply(self.left, self.right.derive(var_name))
        return Add(term1, term2)

```

### **[Markdown Cell]**

## 4. Let it Rip!

Let's test our engine on a linear equation: $y = 3x + 5$.
We know algebraically that the derivative of this line is a constant slope of $3$. Let's see what our engine outputs.

### **[Code Cell]**

```python
x = Variable('x')
eq = (Constant(3) * x) + Constant(5)

print("Original Equation:", eq)

# Let's derive it with respect to 'x'!
derivative = eq.derive('x')

print("Unsimplified Derivative:", derivative)

```

### **[Markdown Cell]**

Look at the output: `(((0 * x) + (3 * 1)) + 0)`.

If you evaluate this mathematically, it simplifies perfectly to $3$!

Our engine doesn't know how to clean up the zeroes and ones yet (that requires a simplification algorithm), but the raw calculus logic is completely, flawlessly correct. You just wrote a program that *understands* calculus.